# 07 — Document Loaders & Text Splitters

The next three notebooks build toward **RAG** (Retrieval-Augmented Generation) — letting a
model answer questions from *your* documents. RAG has two halves: an *ingestion* pipeline
that prepares your data, and a *retrieval + generation* pipeline that uses it at query time.

This notebook covers ingestion's first two steps:

1. **Load** raw sources (text, PDF, web pages) into a standard `Document` object.
2. **Split** those documents into smaller chunks suitable for retrieval.

We'll use the small knowledge base in the `data/` folder (run `python make_data.py` first
if it's missing). It describes a fictional company, *Lumina Robotics* — fictional on
purpose, so later we can prove the model is answering from the documents and not from
memory.

In [ ]:
import os
# Loaders and splitters do not call the LLM, so no API key is needed for this notebook.
assert os.path.exists("data/lumina_handbook.md"), (
    "Knowledge base missing. Run `python make_data.py` in the course folder first."
)
print("Knowledge base found.")

## 1. The `Document` object

Everything you load becomes one or more **`Document`** objects. A `Document` has just two
important attributes:

- **`page_content`** — the text itself (a string).
- **`metadata`** — a dict of information about the source (file path, page number, URL...).

That metadata travels with the text through the whole pipeline, so later you can cite
*where* an answer came from.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Pixel is a home-assistant robot.",
    metadata={"source": "example", "topic": "product"},
)
print(doc.page_content)
print(doc.metadata)

## 2. Loading a text/markdown file with `TextLoader`

`TextLoader` reads a single text file into one `Document`. Loaders return a *list* of
documents (here, a list of one).

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/lumina_handbook.md", encoding="utf-8")
docs = loader.load()

print("Loaded", len(docs), "document(s)")
print("metadata:", docs[0].metadata)
print("first 200 chars:\n", docs[0].page_content[:200])

## 3. Loading a whole folder with `DirectoryLoader`

To ingest many files at once, `DirectoryLoader` walks a folder. You give it a glob
pattern and which loader class to use for each match. Here we load both markdown files.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_loader = DirectoryLoader(
    "data",
    glob="**/*.md",                       # all markdown files, recursively
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
md_docs = dir_loader.load()

print("Loaded", len(md_docs), "markdown documents:")
for d in md_docs:
    print("  -", d.metadata["source"], f"({len(d.page_content)} chars)")

## 4. Loading a PDF with `PyPDFLoader`

PDFs are everywhere in real projects. `PyPDFLoader` (backed by the `pypdf` library) loads
a PDF as **one `Document` per page**, and records the page number in metadata — handy for
citations.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("data/lumina_specs.pdf")
pdf_docs = pdf_loader.load()

print("PDF pages loaded:", len(pdf_docs))
print("metadata of page 0:", pdf_docs[0].metadata)
print("\ncontent preview:\n", pdf_docs[0].page_content[:300])

## 5. Loading a web page with `WebBaseLoader`

`WebBaseLoader` fetches a URL and extracts its text (using BeautifulSoup). This needs an
internet connection; skip it if you're offline. We set a `USER_AGENT` to identify our
requests politely.

In [ ]:
os.environ.setdefault("USER_AGENT", "langchain-course/1.0")
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader("https://python.langchain.com/docs/concepts/")
try:
    web_docs = web_loader.load()
    text = web_docs[0].page_content
    print("Fetched", len(text), "characters from the web page.")
    print(text[:300].strip())
except Exception as e:
    print("Skipping web load (no internet?):", e)

> **There are loaders for almost everything** — CSV, JSON, Notion, Google Drive,
> Slack, YouTube transcripts, SQL, and more — mostly in `langchain_community.document_loaders`.
> They all return the same `Document` list, so the rest of your pipeline doesn't care
> where the data came from.

## 6. Why split documents?

You rarely feed a whole document to the model. You split it into **chunks** because:

- **Context limits & cost** — you can't (and shouldn't) stuff an entire manual into every
  prompt.
- **Retrieval precision** — when a user asks about the warranty, you want to retrieve the
  *paragraph* about warranties, not the whole handbook. Smaller chunks → sharper matches.
- **Embeddings work better on focused text** — a chunk about one topic embeds into a
  cleaner vector than a chunk covering five topics (more on embeddings in notebook 08).

## 7. `RecursiveCharacterTextSplitter` — the default choice

This splitter tries to break text along natural boundaries — paragraphs, then lines, then
sentences, then words — so chunks stay coherent. Two key knobs:

- **`chunk_size`** — the target maximum size of each chunk (in characters by default).
- **`chunk_overlap`** — how many characters consecutive chunks share. Overlap keeps a
  sentence that straddles a boundary from being cut in half and losing meaning.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

chunks = splitter.split_documents(md_docs)
print("Split", len(md_docs), "documents into", len(chunks), "chunks.\n")

for i, c in enumerate(chunks[:3]):
    print(f"--- chunk {i} ({len(c.page_content)} chars, source={os.path.basename(c.metadata['source'])}) ---")
    print(c.page_content[:200].strip(), "...\n")

Notice each chunk **keeps the original metadata** (the source file). That's how, after
retrieval, you can tell the user which document an answer came from.

`split_documents` works on `Document` objects; if you just have a raw string, use
`split_text`, which returns a list of strings.

In [ ]:
sample = md_docs[0].page_content
text_chunks = splitter.split_text(sample)
print("split_text produced", len(text_chunks), "string chunks.")
print("type:", type(text_chunks[0]).__name__)

## 8. Choosing chunk size and overlap

There's no universal answer, but useful guidance:

- **Smaller chunks (200–500 chars)** → precise retrieval, but each chunk has less context;
  an answer might be split across chunks.
- **Larger chunks (1000–2000 chars)** → more context per chunk, but retrieval is fuzzier
  and you spend more tokens per match.
- **Overlap of 10–20% of `chunk_size`** is a sensible starting point.

> **Gotcha — characters are not tokens.** By default `chunk_size` counts *characters*. The
> model's real limit is in *tokens* (~4 chars each in English). For token-accurate
> splitting, build the splitter with `.from_tiktoken_encoder(chunk_size=..., chunk_overlap=...)`
> so the sizes line up with what the model actually counts.

In [ ]:
# Token-aware splitting (sizes are measured in tokens, not characters):
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=128,      # ~128 tokens per chunk
    chunk_overlap=16,
)
token_chunks = token_splitter.split_documents(md_docs)
print("Token-based split produced", len(token_chunks), "chunks.")

## Your turn

Five exercises that mirror building a knowledge base for RAG — a reusable ingestion function,
citable PDF chunks, tagging chunks for filtered retrieval, tuning chunk size, and estimating
embedding cost. Try each in a blank cell before opening its solution.

**Exercise 1 — Reusable ingestion function.** Write `ingest(folder)` that loads every
markdown file in a folder with `DirectoryLoader` and returns split chunks — the one function
you'd call before building any vector store.

*Sample run (illustrative — counts depend on the data files):*

```text
14 chunks ready for a vector store
```

<details><summary>Show solution</summary>

```python
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def ingest(folder: str, chunk_size=500, chunk_overlap=80):
    docs = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

chunks = ingest("data")
print(len(chunks), "chunks ready for a vector store")
```

Adding a PDF or web source later is just another loader feeding the same splitter.

</details>

**Exercise 2 — Citable PDF chunks.** Load `data/lumina_specs.pdf`, split it, and show that
each chunk still carries its **page number** in metadata — the basis for "see page 3"
citations.

*Sample run (illustrative):*

```text
page 0: Lumina Robotics — Pixel Technical Specifications...
page 0: The Pixel Pro ships with a 4-hour battery and five...
page 1: Connectivity: Wi-Fi 6, Bluetooth 5.2, and an optional...
page 1: Warranty: a 24-month limited warranty covers manufacturing...
page 2: Support hours are 9am to 6pm, Monday through Friday...
```

<details><summary>Show solution</summary>

```python
pdf = PyPDFLoader("data/lumina_specs.pdf").load()
pdf_chunks = splitter.split_documents(pdf)
for c in pdf_chunks[:5]:
    print(f"page {c.metadata.get('page')}: {c.page_content[:60].strip()}...")
```

Because the page number rides along in metadata, answers can point back to the exact page.

</details>

**Exercise 3 — Tag chunks for filtered retrieval.** Split the markdown docs, then add a
custom `doc_type` to each chunk's metadata (e.g. `"faq"` vs `"handbook"`) so you can later
restrict a search to one kind of source.

*Sample run (illustrative):*

```text
handbook chunks: 9 of 14
```

<details><summary>Show solution</summary>

```python
chunks = splitter.split_documents(md_docs)
for c in chunks:
    fname = os.path.basename(c.metadata["source"])
    c.metadata["doc_type"] = "faq" if "faq" in fname else "handbook"

handbook = [c for c in chunks if c.metadata["doc_type"] == "handbook"]
print("handbook chunks:", len(handbook), "of", len(chunks))
```

Custom metadata is what lets a vector store filter — "search only the FAQ" later on.

</details>

**Exercise 4 — Tune chunk size.** Split the handbook at `chunk_size=300` and again at `1000`
(with ~15% overlap each) and compare chunk counts and average length. Which favours precise
retrieval, and which favours context?

*Sample run (illustrative):*

```text
chunk_size= 300:  24 chunks, avg 268 chars
chunk_size=1000:   8 chunks, avg 812 chars
```

<details><summary>Show solution</summary>

```python
for size in (300, 1000):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.15))
    ch = sp.split_documents(md_docs)
    avg = sum(len(c.page_content) for c in ch) / len(ch)
    print(f"chunk_size={size:4}: {len(ch):3} chunks, avg {avg:.0f} chars")
```

Small chunks give sharper matches; large chunks give more context per hit but fuzzier search.

</details>

**Exercise 5 — Estimate embedding cost.** Split the docs with the token-aware splitter
(`from_tiktoken_encoder`, ~100 tokens/chunk) and roughly estimate how many tokens it would
take to embed the whole knowledge base.

*Sample run (illustrative):*

```text
31 chunks, ~3050 tokens to embed the knowledge base
```

<details><summary>Show solution</summary>

```python
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=15)
tch = token_splitter.split_documents(md_docs)

total_chars = sum(len(c.page_content) for c in tch)
approx_tokens = total_chars / 4          # ~4 characters per token in English
print(f"{len(tch)} chunks, ~{approx_tokens:.0f} tokens to embed the knowledge base")
```

Knowing the token count up front lets you predict embedding time and cost before you run it.

</details>

## Summary

- Loaders turn raw sources into **`Document`** objects (`page_content` + `metadata`); the
  metadata travels through the whole pipeline for citations.
- `TextLoader`, `DirectoryLoader`, `PyPDFLoader`, and `WebBaseLoader` cover the common
  cases; `langchain_community` has loaders for many more.
- **Split** documents into chunks for context limits, retrieval precision, and better
  embeddings.
- **`RecursiveCharacterTextSplitter`** is the default; tune `chunk_size` and
  `chunk_overlap`, and remember the size is in *characters* unless you use the tiktoken
  encoder variant.

**Next:** [08 — Embeddings & Vector Stores](08_Embeddings_and_Vector_Stores.ipynb), where
we turn these chunks into vectors and search them by meaning.